# Lesson 5A — Data Concepts Playground

**Purpose:** see five data ideas in action on tiny tables (3 customers, 5 loans)
before generating 5,000 synthetic customers in Stage 2.

| # | Concept | One-line meaning |
|---|---|---|
| 1 | Grain | What one row means |
| 2 | Primary key | The column(s) that uniquely identify a row |
| 3 | Composite key | Several columns that are unique only together |
| 4 | Foreign key / orphans | Links between tables, and records with no match |
| 5 | Fan-out | A join that silently copies rows |
| 6 | Roll-up | Aggregating to the right grain BEFORE joining |
| 7 | As-of / leakage | Features may use only the past; the target uses the future |

> All data here is made up for illustration.

## 1. Tables and grain — "what does one row mean?"

- **customers**: one row = one customer
- **loans**: one row = one loan, so customer C1 appears 3 times (3 loans)

**Why it matters:** counts, averages and bad rates are only correct if you know the grain.

In [8]:
# pandas = the library for tables (DataFrames) in Python
import pandas as pd

# A tiny CUSTOMERS table. Grain: ONE ROW = ONE CUSTOMER.
# BAD_FLAG = 1 means this customer later went 30+ DPD (made up for the example)
customers = pd.DataFrame({
    "CUSTOMER_ID": ["C1", "C2", "C3"],
    "SECTOR":      ["Trade", "Agriculture", "Transport"],
    "BAD_FLAG":    [1, 0, 0],
})

# A tiny LOANS table. Grain: ONE ROW = ONE LOAN.
# Notice C1 has THREE loans; C2 and C3 have one each.
loans = pd.DataFrame({
    "LOAN_ID":     ["L1", "L2", "L3", "L4", "L5"],
    "CUSTOMER_ID": ["C1", "C1", "C1", "C2", "C3"],
    "PRODUCT":     ["ASSET_FINANCE", "UNSECURED_BL", "OVERDRAFT", "MORTGAGE", "UNSECURED_BL"],
    "BALANCE":     [2_400_000, 850_000, 300_000, 9_100_000, 1_000_000],   # _ is just a thousands separator for readability
})

# Show both tables (display() prints a nice table in a notebook)
display(customers)
display(loans)

,CUSTOMER_ID,SECTOR,BAD_FLAG
0,C1,Trade,1
1,C2,Agriculture,0
2,C3,Transport,0


,LOAN_ID,CUSTOMER_ID,PRODUCT,BALANCE
0,L1,C1,ASSET_FINANCE,2400000
1,L2,C1,UNSECURED_BL,850000
2,L3,C1,OVERDRAFT,300000
3,L4,C2,MORTGAGE,9100000
4,L5,C3,UNSECURED_BL,1000000


## 2. Primary keys — the unique ID of each row

A **primary key** identifies each row uniquely, like a national ID number.
- `CUSTOMER_ID` is unique in **customers** ✅
- `LOAN_ID` is unique in **loans** ✅
- `CUSTOMER_ID` is **not** unique in loans, and that's correct: one customer can have many loans

**Why it matters:** a key that should be unique but isn't usually means duplicate records.

In [9]:
# Is CUSTOMER_ID unique in customers? Is LOAN_ID unique in loans?
# .is_unique returns True if no value repeats
print("CUSTOMER_ID unique in customers:", customers["CUSTOMER_ID"].is_unique)
print("LOAN_ID unique in loans:        ", loans["LOAN_ID"].is_unique)

# CUSTOMER_ID is NOT unique in loans, and that's correct: one customer can have many loans
print("CUSTOMER_ID unique in loans:    ", loans["CUSTOMER_ID"].is_unique)

CUSTOMER_ID unique in customers: True
LOAN_ID unique in loans:         True
CUSTOMER_ID unique in loans:     False


## 3. Composite keys — catching a duplicate in monthly data

In monthly tables, `LOAN_ID` repeats every month, so the unique key is
**`LOAN_ID` + `SNAPSHOT_DATE` together** (a *composite key*).

A row repeated with the **same loan AND same month** is a real duplicate (planted here: L2 in Jan 2024).

**Why it matters:** a duplicate counts that loan's balance twice in total exposure.

In [10]:
# A tiny LOAN_MONTHLY table. Grain: ONE ROW = ONE LOAN in ONE MONTH.
# I've deliberately planted a DUPLICATE: L2 appears twice for 2024-01-31.
loan_monthly = pd.DataFrame({
    "LOAN_ID":       ["L1",         "L1",         "L1",         "L2",         "L2"],
    "SNAPSHOT_DATE": ["2024-01-31", "2024-02-29", "2024-03-31", "2024-01-31", "2024-01-31"],
    "DPD":           [0,            0,            5,            0,            0],
})
display(loan_monthly)

# LOAN_ID alone repeats (L1 has three months), so it is NOT the key
print("LOAN_ID alone unique:", loan_monthly["LOAN_ID"].is_unique)

# Check the composite key: which rows share the same (LOAN_ID, SNAPSHOT_DATE)?
# duplicated(subset=..., keep=False) marks ALL copies of a repeated key as True
dupes = loan_monthly.duplicated(subset=["LOAN_ID", "SNAPSHOT_DATE"], keep=False)
print("Rows breaking the composite key:")
display(loan_monthly[dupes])

,LOAN_ID,SNAPSHOT_DATE,DPD
0,L1,2024-01-31,0
1,L1,2024-02-29,0
2,L1,2024-03-31,5
3,L2,2024-01-31,0
4,L2,2024-01-31,0


LOAN_ID alone unique: False
Rows breaking the composite key:


,LOAN_ID,SNAPSHOT_DATE,DPD
3,L2,2024-01-31,0
4,L2,2024-01-31,0


## 4. Foreign keys and orphan records

A **foreign key** points to another table's primary key: `loans.CUSTOMER_ID` → `customers.CUSTOMER_ID`.
**Referential integrity** = every loan's customer actually exists.

Here loan **L6** belongs to customer **C9**, who doesn't exist: an **orphan**.

**Why it matters:** orphans silently disappear in joins, so their balances and arrears vanish from portfolio figures.

In [11]:
# Add a loan for customer "C9", who is NOT in the customers table
loans_with_orphan = pd.concat(
    [loans, pd.DataFrame({"LOAN_ID": ["L6"], "CUSTOMER_ID": ["C9"],
                          "PRODUCT": ["MOBILE_LOAN"], "BALANCE": [20_000]})],
    ignore_index=True,   # renumber the rows 0, 1, 2, ...
)

# .isin() asks "is this value in that list?"; ~ means NOT
orphans = ~loans_with_orphan["CUSTOMER_ID"].isin(customers["CUSTOMER_ID"])
print("Loans whose customer does not exist:")
display(loans_with_orphan[orphans])

Loans whose customer does not exist:


,LOAN_ID,CUSTOMER_ID,PRODUCT,BALANCE
5,L6,C9,MOBILE_LOAN,20000


In [12]:
## 5. The join trap — fan-out ⚠️

Joining a one-row-per-customer table to a many-rows-per-customer table **copies** each
customer once per loan.

| | Rows | Bad rate |
|---|---|---|
| Before join (correct) | 3 | 33.3% |
| After join (WRONG) | 5 | 60.0% |

C1 has 3 loans, so its bad flag was counted 3 times, **with no error message**.

**Why it matters:** in a real build with ~10 merged tables, one table with repeated keys
silently corrupts every result after it.

SyntaxError: invalid syntax (1680006891.py, line 3)

## 6. The fix — roll up first, join second, validate always

1. **Roll up** loans to one row per customer (`NO_OF_LOANS`, `TOTAL_BALANCE`)
2. **Join** the rolled-up table to customers
3. **Validate** with `validate="one_to_one"`, so pandas raises an error if keys repeat

Result: still 3 rows, bad rate back to **33.3%** ✅

**This is our project design:** loans rolled up to **customer × month** before joining.

In [ ]:
# Roll loans UP to one row per customer: count the loans, sum the balances
loans_per_customer = (
    loans.groupby("CUSTOMER_ID")                       # one group per customer
         .agg(NO_OF_LOANS=("LOAN_ID", "count"),        # how many loans
              TOTAL_BALANCE=("BALANCE", "sum"))        # total exposure
         .reset_index()                                # turn CUSTOMER_ID back into a normal column
)
display(loans_per_customer)

# Join again. validate="one_to_one" makes pandas RAISE AN ERROR if either side
# has repeated keys, so the fan-out trap can't happen silently.
fixed = customers.merge(loans_per_customer, on="CUSTOMER_ID", how="left", validate="one_to_one")
display(fixed)

print("Rows after correct join:", len(fixed))                         # still 3
print("Bad rate (correct):", round(fixed["BAD_FLAG"].mean(), 3))      # back to 0.333

,CUSTOMER_ID,NO_OF_LOANS,TOTAL_BALANCE
0,C1,3,3550000
1,C2,1,9100000
2,C3,1,1000000


,CUSTOMER_ID,SECTOR,BAD_FLAG,NO_OF_LOANS,TOTAL_BALANCE
0,C1,Trade,1,3,3550000
1,C2,Agriculture,0,1,9100000
2,C3,Transport,0,1,1000000


Rows after correct join: 3
Bad rate (correct): 0.333


## 7. As-of thinking and leakage

Each monthly row is a **photo as at that month-end**.

| Column | Uses | Allowed? |
|---|---|---|
| `MAX_DPD_LAST_3M` | this month + 2 previous | ✅ feature |
| `NEXT_MONTH_DPD__LEAK` | next month (the future) | ❌ leakage |

- **Features** (model inputs) may only use the snapshot month and earlier.
- **The target** ("30+ DPD in the next 3 months") *may* use the future, **only as the label**.
- The last row is empty (`NaN`): no future exists yet, which is why the last 3 months can't be labelled.

**Why it matters:** leakage makes a model look brilliant in testing and fail in real monthly scoring.

In [ ]:
# One customer's DPD over 6 month-ends
history = pd.DataFrame({
    "SNAPSHOT_DATE": pd.date_range("2024-01-31", periods=6, freq="ME"),   # "ME" = month-end dates
    "DPD":           [0, 0, 3, 0, 12, 35],
})

# ✅ LEGITIMATE feature: worst DPD over the last 3 months, including this month
# rolling(3) looks at this row and the 2 rows BEFORE it, never after
history["MAX_DPD_LAST_3M"] = history["DPD"].rolling(3, min_periods=1).max()

# ❌ LEAKAGE: next month's DPD. shift(-1) pulls the value UP from the FUTURE row.
# At scoring time (the month-end) this number does not exist yet.
history["NEXT_MONTH_DPD__LEAK"] = history["DPD"].shift(-1)

display(history)

,SNAPSHOT_DATE,DPD,MAX_DPD_LAST_3M,NEXT_MONTH_DPD__LEAK
0,2024-01-31,0,0.0,0.0
1,2024-02-29,0,0.0,3.0
2,2024-03-31,3,3.0,0.0
3,2024-04-30,0,3.0,12.0
4,2024-05-31,12,12.0,35.0
5,2024-06-30,35,35.0,NaN
